# 🎬 ViralCutter — Google Colab (via Google Drive)
Uma alternativa gratuita ao `opus.pro` e ao `vidyo.ai`

Este notebook roda **seu código local modificado** diretamente do Google Drive,
sem depender do repositório GitHub.

## 📋 Como Usar

1. **Compacte** sua pasta do projeto em um `.zip`
   - ⚠️ **Exclua** as pastas: `.venv`, `VIRALS`, `__pycache__` (para o ZIP ficar leve)
   - No Windows: Botão direito na pasta → *Compactar para arquivo ZIP*
2. **Suba** o `.zip` para o seu Google Drive (qualquer pasta)
3. **Configure** o caminho do ZIP e a pasta de saída na célula abaixo
4. **Execute** as 3 células em ordem (▶️)
5. **Acesse** a WebUI pelo link público que aparecerá no final

> ⚙️ **GPU obrigatória**: Vá em `Runtime → Change runtime type → GPU → T4`

> 💾 Os vídeos gerados serão salvos **automaticamente** no seu Google Drive

In [ ]:
#@title ⚙️ Configuração (Montar Drive + Extrair Projeto)
#@markdown ### 📁 Caminho do ZIP no Google Drive
#@markdown Caminho **relativo ao MyDrive**. Ex: `ViralCutterBETA.zip` ou `Projetos/ViralCutterBETA.zip`
zip_path = "ViralCutterBETA.zip" #@param {type:"string"}
#@markdown ### 💾 Pasta de saída no Drive (vídeos gerados)
#@markdown Os vídeos processados serão salvos nesta pasta do seu Drive.
output_folder = "ViralCutter_Output" #@param {type:"string"}

import os, shutil, glob, zipfile
from google.colab import drive

print("📂 Montando Google Drive...")
drive.mount('/content/drive')

# Caminhos
zip_full = f"/content/drive/MyDrive/{zip_path}"
output_drive = f"/content/drive/MyDrive/{output_folder}"
project_dir = "/content/ViralCutter"

# Verificar ZIP
assert os.path.exists(zip_full), f"❌ ZIP não encontrado em: {zip_full}\nVerifique o caminho e tente novamente."

# Limpar extração anterior
if os.path.exists(project_dir):
    shutil.rmtree(project_dir)
if os.path.exists("/content/_temp_extract"):
    shutil.rmtree("/content/_temp_extract")

# Extrair ZIP
print("📦 Extraindo ZIP...")
with zipfile.ZipFile(zip_full, 'r') as z:
    z.extractall('/content/_temp_extract/')

# Encontrar raiz do projeto automaticamente
matches = glob.glob('/content/_temp_extract/**/main_improved.py', recursive=True)
assert matches, "❌ main_improved.py não encontrado no ZIP! Verifique se compactou a pasta correta."

found_root = os.path.dirname(matches[0])
shutil.move(found_root, project_dir)
shutil.rmtree('/content/_temp_extract/', ignore_errors=True)

# Limpar artefatos do Windows que não funcionam no Linux
cleaned = 0
for root, dirs, files in os.walk(project_dir):
    for d in list(dirs):
        if d in ('__pycache__', '.venv', '.claude'):
            shutil.rmtree(os.path.join(root, d), ignore_errors=True)
            cleaned += 1
if cleaned:
    print(f"🗑️ {cleaned} pasta(s) desnecessária(s) removida(s) (.venv, __pycache__, etc.)")

# Vincular pasta VIRALS ao Google Drive (saída persistente)
os.makedirs(output_drive, exist_ok=True)
virals_local = os.path.join(project_dir, "VIRALS")
if os.path.islink(virals_local):
    os.unlink(virals_local)
elif os.path.exists(virals_local):
    shutil.rmtree(virals_local)
os.symlink(output_drive, virals_local)

print(f"\n✅ Projeto extraído em: {project_dir}")
print(f"💾 Vídeos serão salvos em: Google Drive > {output_folder}")
print(f"\n▶️ Execute a próxima célula para instalar as dependências!")

In [ ]:
#@title 🛠️ Instalação de Dependências
import os
import subprocess
from IPython.display import clear_output

%cd /content/ViralCutter

print("⏳ Iniciando instalação completa...")
print("(Isso pode levar 5-10 minutos na primeira vez)\n")

# 1. UV + Drivers do sistema
print("📦 [1/10] Instalando UV e drivers do sistema...")
subprocess.run(['pip', 'install', 'uv'], check=True)
subprocess.run('sudo apt update -y && sudo apt install -y libcudnn8 ffmpeg xvfb', shell=True, check=True)

# 2. Ambiente Virtual
print("🐍 [2/10] Criando ambiente virtual...")
subprocess.run(['uv', 'venv', '.venv'], check=True)

# 3. PyTorch + CUDA (cu121 — compatível com Colab T4)
print("🔥 [3/10] Instalando PyTorch 2.3.1 + CUDA 12.1...")
cmd_torch = (
    "uv pip install --python .venv "
    "torch==2.3.1+cu121 torchvision==0.18.1+cu121 torchaudio==2.3.1+cu121 "
    "--index-url https://download.pytorch.org/whl/cu121"
)
subprocess.run(cmd_torch, shell=True, check=True)

# 4. WhisperX (do GitHub — versão mais recente com correções de alinhamento)
print("🎤 [4/10] Instalando WhisperX (GitHub)...")
subprocess.run("uv pip install --python .venv git+https://github.com/m-bain/whisperx.git", shell=True, check=True)

# 5. Dependências do requirements.txt
print("📋 [5/10] Instalando requirements.txt...")
subprocess.run("uv pip install --python .venv -r requirements.txt", shell=True, check=True)

# 6. Pacotes extras e correções de compatibilidade
print("🔧 [6/10] Instalando pacotes extras...")
extras = [
    "uv pip install --python .venv yt-dlp pytubefix",
    "uv pip install --python .venv google-generativeai",
    "uv pip install --python .venv pandas",
    "uv pip install --python .venv onnxruntime-gpu",
    "uv pip install --python .venv transformers==4.46.3 'accelerate>=0.26.0'",
]
for cmd in extras:
    subprocess.run(cmd, shell=True, check=True)

# 7. Re-pin PyTorch (garante que nenhuma dependência atualizou sem querer)
print("🔨 [7/10] Re-fixando versão do PyTorch...")
subprocess.run(cmd_torch, shell=True, check=True)

# 8. Travar Numpy e setuptools
print("🔨 [8/10] Travando Numpy...")
subprocess.run("uv pip install --python .venv 'numpy<2.0' setuptools==69.5.1", shell=True, check=True)

# 9. Correção de Visão Computacional (MediaPipe + InsightFace)
print("🔨 [9/10] Corrigindo MediaPipe + InsightFace...")
subprocess.run("uv pip uninstall --python .venv mediapipe protobuf flatbuffers", shell=True)
subprocess.run("uv pip install --python .venv 'mediapipe>=0.10.0' 'protobuf>=3.20,<5.0' 'flatbuffers>=2.0'", shell=True, check=True)
subprocess.run("uv pip install --python .venv insightface onnxruntime-gpu", shell=True, check=True)

# 10. Display virtual (para OpenCV/MediaPipe em modo headless)
print("🖥️ [10/10] Configurando display virtual...")
os.system('Xvfb :1 -screen 0 2560x1440x8 &')
os.environ['DISPLAY'] = ':1.0'

clear_output()
print("✅ Instalação Completa!")
print("━" * 42)
print("  • PyTorch 2.3.1 + CUDA 12.1      ✅")
print("  • WhisperX (GitHub latest)        ✅")
print("  • Transformers 4.46.3             ✅")
print("  • Numpy < 2.0                     ✅")
print("  • MediaPipe + InsightFace          ✅")
print("  • FastAPI + Gradio + Extras        ✅")
print("  • deep-translator + tqdm           ✅")
print("━" * 42)
print("\n▶️ Execute a próxima célula para iniciar a WebUI!")

In [ ]:
#@title 🚀 Executar ViralCutter WebUI
import os

%cd /content/ViralCutter

# Display virtual (caso o kernel tenha reiniciado)
os.system('Xvfb :1 -screen 0 2560x1440x8 &')
os.environ['DISPLAY'] = ':1.0'
os.environ['MPLBACKEND'] = 'Agg'
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
os.environ.setdefault('VC_LIBRARY_MAX_CARDS', '24')

print("🚀 Iniciando ViralCutter WebUI...")
print("⏳ Aguarde o link público (public URL) aparecer abaixo...")
print("⚠️ Ignore avisos de 'UserWarning'\n")

!/content/ViralCutter/.venv/bin/python webui/app.py --colab

---

## Créditos

Inspirado no [reels clips automator](https://github.com/eddieoz/reels-clips-automator) e no [YoutubeVideoToAIPoweredShorts](https://github.com/Fitsbit/YoutubeVideoToAIPoweredShorts)

Desenvolvido por **Rafa.Godoy**

[![GitHub](https://img.shields.io/badge/github-%23121011.svg?style=for-the-badge&logo=github&logoColor=white)](https://github.com/rafaelGodoyEbert)
[![Discord](https://dcbadge.limes.pink/api/server/tAdPHFAbud)](https://discord.gg/tAdPHFAbud)